# E1, E2, E3 — Clasificación con LLMs de Amazon Bedrock

## Preguntas de investigación
- **E1:** ¿un consenso de varios LLMs **sin entrenar** clasifica tan bien como el BERT afinado con 581 ejemplos?
- **E2:** ¿cuál es la mejor forma de combinar los juicios de varios LLMs en una escala **ordinal**?
- **E3:** ¿podemos ampliar el conjunto etiquetado usando los LLMs, sin introducir ruido?

## Método
Usamos 4 modelos de Amazon Bedrock (Nova Micro/Lite/Pro y Llama 3 8B) vía la **Converse API**, con `temperature=0` y un *prompt few-shot* que plantea la escala 1-5 como **aprobación política** (no como estrellas de producto). Comparamos sus predicciones contra la etiqueta humana con QWK/MAE y **bootstrap pareado**.

## Reproducibilidad y su límite (honestidad)
Fijamos el `modelId` con versión y guardamos **todas las respuestas crudas** en `resultados/bedrock_raw.jsonl`, así E1/E2/E3 se recomputan desde ese caché sin volver a llamar a la nube. **Límite honesto:** `temperature=0` NO garantiza determinismo perfecto de un LLM entre versiones del endpoint; lo documentamos como amenaza a la validez.

> Para RE-EJECUTAR las llamadas: `python experimentos/src/classify_581.py` (requiere credenciales Bedrock).

In [1]:
# ============================================================================
# CONFIGURACION COMUN
# ----------------------------------------------------------------------------
# Este bloque prepara el entorno. Se repite en todos los notebooks para que
# cada uno sea autonomo (se pueda abrir y correr por separado).
# ============================================================================
import sys                     # para anadir la carpeta 'src' al path de importacion
import json                    # los resultados de cada experimento se guardan como JSON
from pathlib import Path       # manejo de rutas independiente del sistema operativo

# Anadimos experimentos/src al path para poder importar el codigo compartido
# (metricas ordinales, carga de datos, semillas). Usamos rutas relativas para
# que el notebook funcione sin importar donde este clonado el repositorio.
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np             # calculo numerico (vectores de predicciones)
import pandas as pd            # tablas de resultados legibles
import common as C             # nuestro modulo: metricas ordinales, carga del gold, semilla

# Carpeta donde viven los resultados ya calculados (un JSON por experimento).
R = C.RESULTS

def load(nombre_archivo):
    """Carga un JSON de resultados desde experimentos/resultados/."""
    return json.load(open(R / nombre_archivo))

# --- PATRON DE DOS NIVELES (buena practica de reproducibilidad) ---
# Por defecto RECOMPUTE=False: el notebook CARGA los resultados ya calculados,
# corre en segundos y NO necesita GPU ni Amazon Bedrock. Asi un revisor puede
# abrirlo y ver todo sin infraestructura.
# Si pones RECOMPUTE=True, las celdas marcadas volveran a ENTRENAR/LLAMAR a la nube
# (requiere el venv de ML y/o credenciales de Bedrock; toma minutos a horas).
RECOMPUTE = False

# Fijamos la semilla global para que cualquier calculo aleatorio (p.ej. bootstrap)
# sea reproducible: dos ejecuciones dan el mismo numero.
C.set_all_seeds(C.SEED)
print(f"Entorno listo. Semilla global = {C.SEED}. RECOMPUTE = {RECOMPUTE}.")

Entorno listo. Semilla global = 61298. RECOMPUTE = False.


## E1 — Triple comparación: consenso-LLM vs humano vs BERT

Comparamos dos clasificadores contra la verdad humana, sobre **los mismos 581 comentarios**. El número que importa NO es la diferencia de puntos, sino el **intervalo de confianza de la diferencia**.

In [2]:
# Cargamos el resultado de la triple comparacion.
e1 = load("e1_triple.json")

# Tabla: cada clasificador vs el humano (QWK y MAE; mayor QWK = mejor, menor MAE = mejor).
pd.DataFrame([
    ["BERT+LoRA (afinado, out-of-fold)", round(e1["bert"]["qwk"], 3), round(e1["bert"]["mae"], 3)],
    ["Consenso-LLM (sin entrenar)",       round(e1["llm_consensus"]["qwk"], 3), round(e1["llm_consensus"]["mae"], 3)],
], columns=["clasificador", "QWK", "MAE"])

,clasificador,QWK,MAE
0,"BERT+LoRA (afinado, out-of-fold)",0.415,0.761
1,Consenso-LLM (sin entrenar),0.498,0.639


In [3]:
# La EVIDENCIA es el intervalo de confianza de la diferencia pareada (bootstrap),
# no la resta de los puntos. Si el IC incluye 0, no podemos afirmar diferencia.
d = e1["delta_qwk_llm_minus_bert"]
print(f"Delta QWK (LLM - BERT) = {d['mean']:+.3f}")
print(f"IC 95% de la diferencia = [{d['ci_low']:+.3f}, {d['ci_high']:+.3f}]   (incluye 0)")
print(f"P(LLM > BERT) en el bootstrap = {d['p_llm_gt_bert']:.2f}")
print()
print("CONCLUSION HONESTA: la diferencia NO es concluyente (el IC incluye 0 por un margen minimo).")
print("Enunciado correcto: 'con N=581, el fine-tuning no logra superar a un LLM few-shot'.")

Delta QWK (LLM - BERT) = +0.082
IC 95% de la diferencia = [-0.002, +0.168]   (incluye 0)
P(LLM > BERT) en el bootstrap = 0.97

CONCLUSION HONESTA: la diferencia NO es concluyente (el IC incluye 0 por un margen minimo).
Enunciado correcto: 'con N=581, el fine-tuning no logra superar a un LLM few-shot'.


**Por qué esto es un hallazgo (aunque sea no concluyente):** sugiere que, con datos escasos, invertir en *prompting* de un LLM puede rendir tanto como el costo de entrenar un modelo. Es un resultado útil y defendible para una tesis, siempre que NO se sobrevenda como "victoria".

## E2 — ¿Cómo combinar los juicios de varios LLMs?

Si 4 LLMs dan notas distintas (p. ej. 1, 3, 3, 5), hay que combinarlas. Como la escala **tiene orden**, el esquema de agregación importa.

In [4]:
# Comparamos tres formas de agregar los votos de los 4 LLMs.
e2 = load("e2_aggregation.json")
agg = pd.DataFrame([[k, round(v["qwk"], 3)] for k, v in e2["aggregations"].items()],
                   columns=["agregacion", "QWK"]).sort_values("QWK", ascending=False)
display(agg)
# La mediana respeta el orden y es robusta a un voto atipico (a diferencia de la moda/mayoria).
print(f"Acuerdo inter-LLM (los 4 coinciden): {100*e2['inter_llm_all_agree']:.1f}% de los casos.")
print("=> Solo coinciden en 1 de cada 6 comentarios: la tarea es genuinamente ambigua.")

,agregacion,QWK
1,mediana,0.498
0,mayoria,0.488
2,promedio_red,0.452


Acuerdo inter-LLM (los 4 coinciden): 17.6% de los casos.
=> Solo coinciden en 1 de cada 6 comentarios: la tarea es genuinamente ambigua.


**Lección:** la **mediana ordinal** obtiene el mayor QWK (respeta el orden). No usar voto por mayoría, que trata las clases como categorías sin relación.

## E3 — Ampliar el conjunto etiquetado con auto-etiquetas (resultado nulo, honesto)

Idea: usar los LLMs para etiquetar los ~2000 comentarios sin etiqueta, quedándonos solo con aquellos donde **varios LLMs coinciden** (señal de que es un caso claro). Primero validamos que la concordancia predice la calidad.

In [5]:
# Validacion del "gate" de concordancia: donde mas LLMs coinciden, mas se acercan al humano.
g = load("e3_gate.json")
print("Calidad de la auto-etiqueta segun cuantos LLMs coinciden (medido en los 581 con verdad humana):")
for umbral, v in g["gate_581"].items():
    print(f"  >= {umbral} LLMs coinciden  ->  QWK vs humano = {v['qwk_vs_human']:.3f}")
print()
# Reentrenamos con las auto-etiquetas de alta concordancia y medimos sobre el test humano.
g3 = load("e3_gate3.json"); d = g3["delta_qwk_vs_p0"]
print(f"Con +{g3['n_auto']} auto-etiquetas (gate>=3): QWK = {g3['metrics']['qwk']:.3f}")
print(f"Delta vs baseline = {d['mean']:+.3f}  IC[{d['ci_low']:+.3f},{d['ci_high']:+.3f}]  -> dentro del ruido")

Calidad de la auto-etiqueta segun cuantos LLMs coinciden (medido en los 581 con verdad humana):
  >= 2 LLMs coinciden  ->  QWK vs humano = 0.496
  >= 3 LLMs coinciden  ->  QWK vs humano = 0.577
  >= 4 LLMs coinciden  ->  QWK vs humano = 0.792

Con +1210 auto-etiquetas (gate>=3): QWK = 0.449
Delta vs baseline = +0.033  IC[-0.025,+0.089]  -> dentro del ruido


## Veredicto de E1-E3 y amenazas
- **E1:** el LLM iguala al BERT (no concluyente, pero interesante).
- **E2:** usar mediana, no mayoría.
- **E3:** resultado **nulo** con causa entendida: los 4 LLMs solo coinciden en 17.6% de los casos, así que las auto-etiquetas favorecen los comentarios fáciles y **estrechan la distribución** (sesgo de selección). Es un riesgo de propagación de sesgo que documentamos; por eso reportamos E3 como nulo, no como mejora.